# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We use the Croissant schema to dynamically discover record sets, fields, and values.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using [`mlcroissant`](https://github.com/mlcommons/croissant).

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset metadata and parse as object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', 'Unnamed')}")
print(f"Description: {getattr(metadata, 'description', '')[:200]}...")

## 2. Data Overview
Review available record sets, fields, and their `@id`s (identifiers).

Let's list all record sets, then for each, show its fields. All referencing uses `@id`.

In [ ]:
# Get all record set objects (they have '@type' == 'cr:RecordSet')
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    # If the JSON-LD had 'recordSet' key, not 'record_sets'
    record_sets = getattr(metadata, 'recordSet')

# If record_sets are not already a list of object, try dataset.record_sets()
if not record_sets:
    record_sets = list(dataset.record_sets())

print(f"Found {len(record_sets)} record set(s).")
record_set_ids = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    rs_name = getattr(rs, 'name', getattr(rs, '@id', ''))
    print(f"\nRecord Set: {rs_name} (ID: {rs_id})")
    record_set_ids.append(rs_id)
    # Fields for this record set
    fields = getattr(rs, 'fields', []) or getattr(rs, 'field', [])
    for field in fields:
        field_id = getattr(field, '@id', '')
        field_name = getattr(field, 'name', field_id)
        print(f"    Field: {field_name} (ID: {field_id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as above.

We will attempt to load *all* record sets as available, using their `@id` for extraction.

In [ ]:
# We'll iterate through each record set using its @id and load records as DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print("Fields:", list(df.columns))
        print(df.head(2))
    else:
        print("No records found.")

# For further steps, select the first non-empty record set
nonempty_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        nonempty_record_set_id = rsid
        break
if nonempty_record_set_id is not None:
    print(f"\nDefault record set for analysis: {nonempty_record_set_id}")
else:
    print("No non-empty record sets found!")

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps: filtering, normalization, grouping. All references use `@id` as the field or record set.

*Please edit the field IDs below as appropriate for the columns in your selected record set.*

In [ ]:
# Pick the record set for further analysis
record_set_id = nonempty_record_set_id
df = dataframes[record_set_id]

# Display columns with their IDs
print("Columns in selected DataFrame:")
for col in df.columns:
    print(f"  - {col}")

# Pick a numeric field ID (column) for analysis
numeric_field_id = None
for col in df.columns:
    # Heuristics: choose a field that sounds numeric, fallback to the first one
    if any(s in col.lower() for s in ['age', 'interval', 'years', 'duration', 'count', 'score']):
        numeric_field_id = col
        break
if numeric_field_id is None and len(df.columns) > 0:
    numeric_field_id = df.columns[0]

print(f"\nNumeric field ID selected: {numeric_field_id}")

# Convert to numeric if needed (errors='coerce' turns bad data to NaN)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: keep rows with numeric_field > threshold
threshold = df[numeric_field_id].dropna().quantile(0.75)  # e.g., 75th percentile
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    (filtered_df[numeric_field_id].std() + 1e-9)
)

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g., 'Sex', 'MSI status', etc.)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < max(10, int(0.1 * len(df))):
        group_field_id = col
        break

if group_field_id is not None:
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    print("No appropriate categorical field found to group by.")

## 5. Visualization
Visualize data distributions and relationships between fields using `matplotlib` and `seaborn` (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field is available, make a boxplot
if group_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded FAIR² dataset metadata via Croissant and inspected its structure
- Dynamically retrieved available record sets and fields by `@id`
- Extracted data using the record set and field identifiers
- Demonstrated basic filtering, normalization, EDA, and grouping using only IDs
- Visualized structure and distributions of a selected numeric variable

This approach facilitates reproducible data exploration for any Croissant-format dataset, referencing all entities by their unique `@id`.
